In [ ]:
import os

os.environ["GROQ_API_KEY"] = "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"

In [ ]:
!pip install -q langchain-groq langchain-core requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.0 MB/s eta 0:00:00


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [ ]:
# tool create

@tool
def multiply(a:int, b:int) -> int:
  """return the product of two numbers"""
  return a * b

In [ ]:
print(multiply.invoke({'a':3, 'b':4}))

12


In [ ]:
# Tool Binding

llm = ChatGroq(
    model="llama-3.1-8b-instant"
)

In [ ]:
llm_with_tools = llm.bind_tools([multiply])  # => This is what a tool binding is...

In [ ]:
llm_with_tools.invoke('Hi how are you?')

AIMessage(content="I'm functioning properly, thank you for asking. Is there anything I can help you with today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 223, 'total_tokens': 244, 'completion_time': 0.035233299, 'completion_tokens_details': None, 'prompt_time': 0.014482037, 'prompt_tokens_details': None, 'queue_time': 0.171074332, 'total_time': 0.049715336}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0c2e-1a79-7cf2-9f8d-b7f69d57bc53-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 223, 'output_tokens': 21, 'total_tokens': 244})

In [ ]:
llm_with_tools.invoke('can you multiply 3 with 10')

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'pgr37rxkh', 'function': {'arguments': '{"a":3,"b":10}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 226, 'total_tokens': 245, 'completion_time': 0.031309109, 'completion_tokens_details': None, 'prompt_time': 0.020811981, 'prompt_tokens_details': None, 'queue_time': 0.067960504, 'total_time': 0.05212109}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0c2e-9011-7351-bb66-cf08a5c42791-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'pgr37rxkh', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 226, 'output_tokens': 19, 'total_tokens': 245})

In [ ]:
llm_with_tools.invoke('can you multiply 3 with 10').tool_calls[0]

{'name': 'multiply',
 'args': {'a': 3, 'b': 10},
 'id': '0kx7bthfj',
 'type': 'tool_call'}

In [ ]:
# Tool Execution
result = llm_with_tools.invoke('can you multiply 3 with 10')

In [ ]:
result.tool_calls[0]['args']

{'a': 3, 'b': 10}

In [ ]:
# Giving only args as the input to tool
multiply.invoke(
    result.tool_calls[0]['args']
)

30

In [ ]:
# Giving the complete tool call as the input
multiply.invoke(
    result.tool_calls[0]
)

# Getting the ToolMessage as the output

ToolMessage(content='30', name='multiply', tool_call_id='r8zmbgd34')

Dynamic Execution with changing query changing the output

In [ ]:
query = HumanMessage('can you multiply 3 with 1')

In [ ]:
message = [query]

In [ ]:
message

[HumanMessage(content='can you multiply 3 with 1', additional_kwargs={}, response_metadata={})]

In [ ]:
result = llm_with_tools.invoke(message)

In [ ]:
message.append(result)

In [ ]:
message

[HumanMessage(content='can you multiply 3 with 1', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'xjgdm8wnj', 'function': {'arguments': '{"a":3,"b":1}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 226, 'total_tokens': 245, 'completion_time': 0.036588471, 'completion_tokens_details': None, 'prompt_time': 0.016571562, 'prompt_tokens_details': None, 'queue_time': 0.13712719, 'total_time': 0.053160033}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_d9492c3c54', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0c42-1d55-7920-ab1b-8853a6e39d7e-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1}, 'id': 'xjgdm8wnj', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 226, 'output_tokens': 19, 'total_tokens': 245})]

In [ ]:
tool_result = multiply.invoke(
    result.tool_calls[0]
)

In [ ]:
message.append(tool_result)

In [ ]:
message

[HumanMessage(content='can you multiply 3 with 1', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'xjgdm8wnj', 'function': {'arguments': '{"a":3,"b":1}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 226, 'total_tokens': 245, 'completion_time': 0.036588471, 'completion_tokens_details': None, 'prompt_time': 0.016571562, 'prompt_tokens_details': None, 'queue_time': 0.13712719, 'total_time': 0.053160033}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_d9492c3c54', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0c42-1d55-7920-ab1b-8853a6e39d7e-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1}, 'id': 'xjgdm8wnj', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 226, 'output_tokens': 19, 'total_tokens': 245}),
 ToolMessage(cont

In [ ]:
llm_with_tools.invoke(message).content

'The result of the multiplication of 3 and 1 is 3.'

## Currancy Conversion Tool

In [ ]:
from typing import Annotated
from langchain_core.tools import InjectedToolArg, tool # To tackle such situation we use this InjectedToolArg
import requests

# tool create

@tool
def get_conversion_factor(base_curr: str, target_curr: str) -> float:
  """This function fetces the currancy conversion factor between a give base currency and a target currency"""

  url = f"https://v6.exchangerate-api.com/v6/1858dae0317154cc1f6a6e40/latest/{base_curr}"

  res = requests.get(url)
  data = res.json()

  return data["conversion_rates"][target_curr]


@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate

In [ ]:
get_conversion_factor.invoke({'base_curr': 'USD', 'target_curr': 'INR'})

93.5667

In [ ]:
convert.invoke({'base_currency_value': 32, 'conversion_rate': 93.5667})

2994.1344

In [ ]:
# Tool Binding

llm = ChatGroq(
    model="llama-3.1-8b-instant"
)

In [ ]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [ ]:
message = [HumanMessage('What is the conversion factor between USD and INR, and based on that can you convert 32 usd into inr')]

In [ ]:
message

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 32 usd into inr', additional_kwargs={}, response_metadata={})]

In [ ]:
ai_message = llm_with_tools.invoke(message)

In [ ]:
message.append(ai_message)

In [ ]:
ai_message.tool_calls

# Here you see the conversion_factor is 3.55 which is says that our llm gets confused with our structure so to handle such situation you can use the INJECTED Tool Arguments

[{'name': 'get_conversion_factor',
  'args': {'base_curr': 'USD', 'target_curr': 'INR'},
  'id': 'hkdrrvsg3',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 32, 'conversion_factor': 3.55},
  'id': '2g2ekpzs1',
  'type': 'tool_call'}]

In [ ]:
# Note => If you make any argument to injected tool argument then llm not set that value
# during the tool calling, It's you to set that value

In [ ]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(
        tool_call
    )
    # fetch this message rate
    conversion_rate = float(tool_message1.content)

    # Appen it
    message.append(tool_message1)

  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    message.append(tool_message2)

In [ ]:
llm_with_tools.invoke(message).content

'The conversion factor between USD and INR is 93.5667, and 32 USD is equivalent to 2994.1344 INR.'